# 03 — Content Comparator: TF-IDF Cosine Similarity + Edit Distance

Companion notebook to `03-content-comparison-and-similarity.md`.

Implements the layered approach the real Content Comparator tool would use: a fast TF-IDF cosine
similarity filter across an "approved claims" library, plus a from-scratch Levenshtein edit-distance
check (and `difflib` as a standard-library alternative) for near-verbatim-quote checking. New
claims are compared against the library and flagged as near-duplicates/unsupported when they fall
above/below configurable thresholds.

Fully offline — only `scikit-learn` and the standard library `difflib`. No internet, no GPU.


## 1. Approved claims library and new claims to check

In [1]:
approved_claims_library = [
    "Drug X reduces symptom severity by 42% compared to placebo.",
    "The most common adverse reaction was mild headache.",
    "The recommended starting dose of Drug X is 10mg once daily.",
    "Drug X is more effective than Drug Y at reducing flare frequency.",
    "Drug X should not be used in patients with severe liver impairment.",
    "Clinical trials demonstrate a 35% reduction in relapse rates.",
    "Drug X carries a boxed warning for increased cardiovascular risk.",
]

# Candidate new claims from a piece of marketing content under review.
new_claims_to_check = [
    # near-verbatim match (should flag as duplicate/supported)
    "Drug X reduces symptom severity by 42% versus placebo.",
    # paraphrase of an approved claim using different vocabulary
    "Drug X lowers the intensity of symptoms compared with a sugar pill.",
    # legitimate restatement of the dosing claim
    "Patients should start Drug X at a 10mg once-daily dose.",
    # an UNSUPPORTED claim -- no approved source says this
    "Drug X completely eliminates all symptoms within 24 hours.",
    # near-verbatim quote of a safety claim, should match cleanly
    "The most common adverse reaction reported was mild headache.",
]

print(f"Approved library: {len(approved_claims_library)} claims")
print(f"New claims to check: {len(new_claims_to_check)} claims")


Approved library: 7 claims
New claims to check: 5 claims


## 2. TF-IDF cosine similarity (fast first-pass filter)

Fit one TF-IDF vectorizer across the combined vocabulary of the library and the new claims, then
compute cosine similarity between every new claim and every approved claim. This scales to a large
library because it's just a sparse matrix multiply.


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")
vectorizer.fit(approved_claims_library + new_claims_to_check)

library_vectors = vectorizer.transform(approved_claims_library)
new_vectors = vectorizer.transform(new_claims_to_check)

tfidf_sim_matrix = cosine_similarity(new_vectors, library_vectors)
print(f"Similarity matrix shape: {tfidf_sim_matrix.shape}  (new_claims x approved_library)")


Similarity matrix shape: (5, 7)  (new_claims x approved_library)


## 3. Edit distance: from-scratch Levenshtein + difflib

Levenshtein distance implemented directly (no external dependency needed), plus `difflib`'s
`SequenceMatcher.ratio()` as a standard-library alternative similarity score (0-1, higher = more
similar), useful for catching near-verbatim quote drift.


In [3]:
import difflib


def levenshtein(a: str, b: str) -> int:
    """Minimum single-character edits (insert/delete/substitute) to turn a into b."""
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,       # deletion
                dp[i][j - 1] + 1,       # insertion
                dp[i - 1][j - 1] + cost,  # substitution
            )
    return dp[m][n]


def levenshtein_similarity(a: str, b: str) -> float:
    """Normalize edit distance into a 0-1 similarity score (1 = identical)."""
    max_len = max(len(a), len(b))
    if max_len == 0:
        return 1.0
    return 1.0 - (levenshtein(a, b) / max_len)


def difflib_similarity(a: str, b: str) -> float:
    return difflib.SequenceMatcher(None, a, b).ratio()


# Sanity check against a known near-identical pair
a = "Drug X reduces symptom severity by 42% compared to placebo."
b = "Drug X reduces symptom severity by 42% versus placebo."
print(f"Levenshtein distance: {levenshtein(a, b)}")
print(f"Levenshtein similarity: {levenshtein_similarity(a, b):.3f}")
print(f"difflib similarity:     {difflib_similarity(a, b):.3f}")


Levenshtein distance: 10
Levenshtein similarity: 0.831
difflib similarity:     0.867


## 4. The comparator: layer TF-IDF cosine + edit distance, flag near-duplicates

For each new claim: find its best-matching approved claim by TF-IDF cosine similarity, then compute
edit-distance similarity against that specific best match. Flag as:
- **supported / near-duplicate** if TF-IDF cosine similarity is high (a paraphrase or restatement of
  something approved)
- **unsupported** if no approved claim clears the similarity threshold at all


In [4]:
TFIDF_FLAG_THRESHOLD = 0.35   # below this, no approved claim is considered a real match
EDIT_SIM_NOTE_THRESHOLD = 0.85  # above this, flag as "near-verbatim" (relevant for quote-accuracy checks)

results = []
for i, new_claim in enumerate(new_claims_to_check):
    sims = tfidf_sim_matrix[i]
    best_idx = sims.argmax()
    best_score = sims[best_idx]
    best_match = approved_claims_library[best_idx]
    edit_sim = levenshtein_similarity(new_claim, best_match)

    if best_score < TFIDF_FLAG_THRESHOLD:
        status = "UNSUPPORTED -- flag for review"
    elif edit_sim >= EDIT_SIM_NOTE_THRESHOLD:
        status = "near-verbatim match (supported)"
    else:
        status = "paraphrase of an approved claim (supported)"

    results.append({
        "new_claim": new_claim,
        "best_match": best_match if best_score >= TFIDF_FLAG_THRESHOLD else None,
        "tfidf_cosine": round(float(best_score), 3),
        "edit_similarity": round(edit_sim, 3),
        "status": status,
    })

for r in results:
    print(f"NEW CLAIM: {r['new_claim']}")
    print(f"  best match:      {r['best_match']}")
    print(f"  TF-IDF cosine:   {r['tfidf_cosine']}")
    print(f"  edit similarity: {r['edit_similarity']}")
    print(f"  STATUS: {r['status']}")
    print()


NEW CLAIM: Drug X reduces symptom severity by 42% versus placebo.
  best match:      Drug X reduces symptom severity by 42% compared to placebo.
  TF-IDF cosine:   0.817
  edit similarity: 0.831
  STATUS: paraphrase of an approved claim (supported)

NEW CLAIM: Drug X lowers the intensity of symptoms compared with a sugar pill.
  best match:      None
  TF-IDF cosine:   0.182
  edit similarity: 0.388
  STATUS: UNSUPPORTED -- flag for review

NEW CLAIM: Patients should start Drug X at a 10mg once-daily dose.
  best match:      The recommended starting dose of Drug X is 10mg once daily.
  TF-IDF cosine:   0.563
  edit similarity: 0.407
  STATUS: paraphrase of an approved claim (supported)

NEW CLAIM: Drug X completely eliminates all symptoms within 24 hours.
  best match:      None
  TF-IDF cosine:   0.08
  edit similarity: 0.262
  STATUS: UNSUPPORTED -- flag for review

NEW CLAIM: The most common adverse reaction reported was mild headache.
  best match:      The most common adverse reac

## 5. Summary table


In [5]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df


,new_claim,best_match,tfidf_cosine,edit_similarity,status
0,Drug X reduces symptom severity by 42% versus ...,Drug X reduces symptom severity by 42% compare...,0.817,0.831,paraphrase of an approved claim (supported)
1,Drug X lowers the intensity of symptoms compar...,NaN,0.182,0.388,UNSUPPORTED -- flag for review
2,Patients should start Drug X at a 10mg once-da...,The recommended starting dose of Drug X is 10m...,0.563,0.407,paraphrase of an approved claim (supported)
3,Drug X completely eliminates all symptoms with...,NaN,0.080,0.262,UNSUPPORTED -- flag for review
4,The most common adverse reaction reported was ...,The most common adverse reaction was mild head...,0.887,0.850,near-verbatim match (supported)


## Takeaways

- TF-IDF cosine similarity alone catches the near-verbatim and lightly-reworded cases well
  ("versus placebo" vs. "compared to placebo") because they share most of their distinctive
  vocabulary.
- It's weaker on genuine paraphrases with different vocabulary ("lowers the intensity of symptoms
  compared with a sugar pill" vs. "reduces symptom severity ... compared to placebo") — a
  production system would add a sentence-embedding similarity layer (chapter 03) on the shortlist to
  catch these more reliably; this notebook intentionally keeps to TF-IDF + edit distance to stay
  fully offline with no extra downloads.
- The unsupported claim ("completely eliminates all symptoms within 24 hours") correctly falls
  below every threshold — nothing in the approved library says anything like that, which is exactly
  the case the Content Comparator exists to catch.
- Thresholds (`TFIDF_FLAG_THRESHOLD`, `EDIT_SIM_NOTE_THRESHOLD`) are business/compliance decisions
  tuned against labeled examples, not fixed constants — see the chapter's discussion on
  precision/recall trade-offs and biasing toward recall in a compliance context.
